# 01 — CuPy: NumPy on the GPU

> CuPy gives you the full NumPy/SciPy API, running on the GPU.  
> The mental model: **arrays live on the GPU, operations execute on the GPU.**

---

## Table of Contents

1. [Core Concepts](#1-core-concepts)
2. [Array Creation and Transfer](#2-array-creation-and-transfer)
3. [Operations](#3-operations)
4. [Performance and Timing](#4-performance-and-timing)
5. [Memory Management](#5-memory-management)
6. [Custom Kernels in CuPy](#6-custom-kernels-in-cupy)
7. [SciPy on GPU — cupyx.scipy](#7-scipy-on-gpu--cupyxscipy)
8. [FFT and Linear Algebra](#8-fft-and-linear-algebra)
9. [Interoperability](#9-interoperability)
10. [Common Pitfalls](#10-common-pitfalls)
11. [Exercises](#11-exercises)

---

## 1. Core Concepts

### What CuPy does

CuPy wraps CUDA libraries (cuBLAS, cuFFT, cuSolver, cuSPARSE, cuDNN) behind a NumPy-compatible API. When you write `cp.dot(A, B)`, CuPy calls cuBLAS's `sgemm` — the same routine that powers PyTorch and TensorFlow matmuls.

You do **not** write GPU kernels with CuPy in the normal workflow. The library's built-in ops cover the vast majority of numerical computing needs.

### The `np` → `cp` rule

For most code, replacing `import numpy as np` with `import cupy as cp` is all you need. The APIs are intentionally identical.

In [ ]:
# CPU version
import numpy as np
result = np.fft.fft(np.random.rand(1024))

# GPU version — identical code, different import
import cupy as cp
result = cp.fft.fft(cp.random.rand(1024))   # runs on GPU

### What CuPy does NOT do

- CuPy cannot run arbitrary Python code on the GPU. Only NumPy-style operations.
- CuPy does not JIT-compile Python functions into GPU kernels (that's Numba/Triton).
- CuPy arrays are **not** NumPy arrays. You cannot mix them in a single operation.

---

## 2. Array Creation and Transfer

### Creating arrays

In [ ]:
import cupy as cp
import numpy as np

# --- Direct creation on GPU ---
a = cp.array([1, 2, 3, 4], dtype=cp.float32)     # from Python list
z = cp.zeros((512, 512), dtype=cp.float32)         # zeros
o = cp.ones((1024,), dtype=cp.float64)             # ones
r = cp.random.rand(1000, 1000).astype(cp.float32)  # random
li = cp.linspace(0, 2 * cp.pi, 100)               # linspace
ar = cp.arange(0, 100, 2, dtype=cp.int32)         # arange

# --- From NumPy (host → device) ---
np_arr = np.random.rand(1000).astype(np.float32)
cp_arr = cp.asarray(np_arr)          # copies to GPU

# --- Back to NumPy (device → host) ---
back = cp.asnumpy(cp_arr)            # copies to CPU
back = cp_arr.get()                  # same thing, alternative syntax

### Understanding the memory locations

In [ ]:
a_cpu = np.zeros(10)     # RAM (host)
a_gpu = cp.zeros(10)     # VRAM (device)

print(a_cpu.device)      # CPU
print(a_gpu.device)      # <CUDA Device 0>
print(a_gpu.nbytes)      # 40 bytes (10 * float64)

cpu
<CUDA Device 0>
80


### Transfer costs

Data transfer between CPU and GPU is expensive — typically 10–20 GB/s on PCIe 4.0, compared to 700+ GB/s GPU memory bandwidth. Design your code to minimize round trips.

In [ ]:
import time

N = 1 << 25  # 32M floats = 128 MB

a = np.random.rand(N).astype(np.float32)

# Measure H→D transfer
t0 = time.perf_counter()
d_a = cp.asarray(a)
cp.cuda.Stream.null.synchronize()
t_h2d = (time.perf_counter() - t0) * 1e3

# Measure D→H transfer
t0 = time.perf_counter()
back = cp.asnumpy(d_a)
t_d2h = (time.perf_counter() - t0) * 1e3

print(f"H→D: {t_h2d:.1f} ms  ({128/t_h2d*1e3:.1f} MB/s effective)")
print(f"D→H: {t_d2h:.1f} ms  ({128/t_d2h*1e3:.1f} MB/s effective)")

H→D: 36.0 ms  (3552.4 MB/s effective)
D→H: 48.6 ms  (2635.1 MB/s effective)


---

## 3. Operations

### Arithmetic and broadcasting

In [ ]:
a = cp.array([1., 2., 3., 4.], dtype=cp.float32)
b = cp.array([10., 20., 30., 40.], dtype=cp.float32)

print(a + b)          # [11, 22, 33, 44]
print(a * b)          # element-wise multiply
print(a ** 2)         # [1, 4, 9, 16]
print(cp.sqrt(a))     # [1, 1.41, 1.73, 2]

# Broadcasting (identical to NumPy)
A = cp.ones((4, 3))
v = cp.array([1., 2., 3.])
print((A * v).shape)  # (4, 3) — v broadcast across rows

[11. 22. 33. 44.]
[ 10.  40.  90. 160.]
[ 1.  4.  9. 16.]
[1.        1.4142135 1.7320508 2.       ]
(4, 3)


### Reductions

In [ ]:
x = cp.random.rand(1000, 1000).astype(cp.float32)

print(cp.sum(x))                    # total sum
print(cp.sum(x, axis=0).shape)      # (1000,) — column sums
print(cp.sum(x, axis=1).shape)      # (1000,) — row sums
print(cp.mean(x))
print(cp.std(x))
print(cp.min(x), cp.max(x))
print(cp.argmin(x), cp.argmax(x))

499862.1
(1000,)
(1000,)
0.4998621
0.2888133
1.4304949e-06 0.99999976
441055 80214


### Sorting and searching

In [ ]:
a = cp.array([3, 1, 4, 1, 5, 9, 2, 6])

print(cp.sort(a))                   # [1, 1, 2, 3, 4, 5, 6, 9]
print(cp.argsort(a))                # indices that sort a
print(cp.where(a > 3))              # indices where condition is true
print(cp.unique(a))                 # [1, 2, 3, 4, 5, 6, 9]

[1 1 2 3 4 5 6 9]
[1 3 6 0 2 4 7 5]
(array([2, 4, 5, 7]),)
[1 2 3 4 5 6 9]


### Boolean indexing

In [ ]:
a = cp.random.randn(100).astype(cp.float32)
positive = a[a > 0]    # works just like NumPy
print(positive.shape)  # (roughly 50,)

(50,)


---

## 4. Performance and Timing

### The synchronization rule

**GPU operations are asynchronous.** When you call `cp.sum(x)`, the GPU starts working but Python continues immediately. To measure real GPU time you must synchronize:

In [ ]:
import time
import cupy as cp

N = 1 << 24
x = cp.random.rand(N, dtype=cp.float32)

# WRONG: measures scheduling overhead, not GPU time
t0 = time.perf_counter()
cp.sum(x)
t_wrong = (time.perf_counter() - t0) * 1e3
print(f"Wrong measurement: {t_wrong:.4f} ms")  # will be suspiciously fast

# CORRECT: synchronize before and after
cp.cuda.Stream.null.synchronize()
t0 = time.perf_counter()
cp.sum(x)
cp.cuda.Stream.null.synchronize()
t_correct = (time.perf_counter() - t0) * 1e3
print(f"Correct measurement: {t_correct:.4f} ms")

Wrong measurement: 0.2975 ms
Correct measurement: 0.4328 ms


### When GPU wins

The GPU has much higher memory bandwidth and compute throughput than the CPU, but it has significant **launch overhead** (~10–50 µs per kernel). Small arrays don't benefit:

In [ ]:
import numpy as np

def benchmark(n):
    a_np = np.random.rand(n).astype(np.float32)
    a_cp = cp.asarray(a_np)

    # CPU
    t0 = time.perf_counter()
    for _ in range(10): np.sum(a_np)
    t_cpu = (time.perf_counter() - t0) / 10 * 1e3

    # GPU
    cp.cuda.Stream.null.synchronize()
    t0 = time.perf_counter()
    for _ in range(10):
        cp.sum(a_cp)
        cp.cuda.Stream.null.synchronize()
    t_gpu = (time.perf_counter() - t0) / 10 * 1e3

    return t_cpu, t_gpu

print(f"{'N':>12} | {'CPU (ms)':>10} | {'GPU (ms)':>10} | {'Speedup':>8}")
print("-" * 50)
for exp in [10, 14, 18, 22, 25]:
    t_cpu, t_gpu = benchmark(1 << exp)
    print(f"{1<<exp:>12,} | {t_cpu:>10.3f} | {t_gpu:>10.3f} | {t_cpu/t_gpu:>7.1f}x")

           N |   CPU (ms) |   GPU (ms) |  Speedup
--------------------------------------------------
       1,024 |      0.017 |      0.068 |     0.3x
      16,384 |      0.030 |      0.057 |     0.5x
     262,144 |      0.076 |      0.041 |     1.8x
   4,194,304 |      1.103 |      0.104 |    10.6x
  33,554,432 |     12.438 |      0.545 |    22.8x


In [ ]:
def bench_cpu(fn, *args, repeats=5):
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn(*args)
        times.append((time.perf_counter() - t0) * 1e3)
    return min(times)  # ms

def bench_gpu_cp(fn, *args, repeats=5):
    times = []
    for _ in range(repeats):
        cp.cuda.Stream.null.synchronize()
        t0 = time.perf_counter()
        fn(*args)
        cp.cuda.Stream.null.synchronize()
        times.append((time.perf_counter() - t0) * 1e3)
    return min(times)  # ms

print("Timing helpers ready.")

Timing helpers ready.


Expected output (approximate, T4 GPU):

```
           N |   CPU (ms) |   GPU (ms) |  Speedup
--------------------------------------------------
       1,024 |      0.003 |      0.050 |    0.1x   ← GPU loses
      16,384 |      0.008 |      0.052 |    0.2x   ← GPU loses
     262,144 |      0.120 |      0.060 |    2.0x   ← break-even
   4,194,304 |      1.800 |      0.110 |   16.4x   ← GPU wins
  33,554,432 |     14.500 |      0.720 |   20.1x   ← GPU dominates
```

**Rule:** For arrays smaller than ~100K elements, GPU overhead dominates. For large arrays, GPU wins.

---

## 5. Memory Management

### The memory pool

CuPy uses a **memory pool** to avoid the overhead of `cudaMalloc` on every allocation. Memory freed with `del` returns to the pool, not to the OS.

In [ ]:
pool = cp.get_default_memory_pool()

def show_mem(label):
    u = pool.used_bytes() / 1024**2
    t = pool.total_bytes() / 1024**2
    print(f"[{label}] used={u:.1f} MB, total={t:.1f} MB")

show_mem("start")

x = cp.zeros((1024, 1024, 16), dtype=cp.float32)   # allocate ~64 MB
show_mem("allocated 64 MB")

del x                           # returns to pool
show_mem("del (pool keeps it)")

pool.free_all_blocks()          # actually release to OS
show_mem("free_all_blocks")

[start] used=196.8 MB, total=396.5 MB
[allocated 64 MB] used=196.8 MB, total=396.5 MB
[del (pool keeps it)] used=132.8 MB, total=396.5 MB
[free_all_blocks] used=132.8 MB, total=132.9 MB


### CuPy Memory Pool — Output Explained
---

#### `[start]` — used=196.8 MB, total=352.3 MB

Before your code runs, CuPy (and the Colab GPU runtime itself) already occupies ~197 MB. This is baseline overhead — CUDA context, CuPy internals, etc. Nothing you allocated yet.

---

#### `[allocated 64 MB]` — used=196.8 MB, total=352.3 MB

You just allocated a 64 MB array, but `used` didn't increase by 64 MB. This is because CuPy's memory pool **pre-allocates a large chunk upfront** (352 MB total) and serves allocations from that pool. Your 64 MB was already inside the pre-reserved pool, so `used` doesn't visibly jump — it was already counted in `total`.

---

#### `[del (pool keeps it)]` — used=132.8 MB, total=352.3 MB

After `del big`, `used` drops by ~64 MB (196.8 → 132.8). But `total` stays at 352.3 MB. This is the key behavior of CuPy's memory pool:

- `del` in Python does **not** call `cudaFree`
- The memory is returned to the pool (marked as available) but stays reserved on the GPU
- This is intentional — `cudaMalloc` is expensive (~milliseconds). By keeping the memory reserved, the next allocation is instant

---

#### `[free_all_blocks]` — used=132.8 MB, total=132.9 MB

After calling `pool.free_all_blocks()`, the pool actually releases all unused blocks back to the OS via `cudaFree`. Now `total` drops from 352.3 MB to 132.9 MB — matching `used`. Only the memory that is actively in use remains reserved.

---

#### The Big Picture

```
total = memory reserved from the GPU (cudaMalloc'd by the pool)
used  = memory currently handed out to your arrays
free  = total - used  (available in pool for next allocation)
```

| Action | used | total | Why |
|--------|------|-------|-----|
| Start | 196.8 | 352.3 | Baseline runtime overhead |
| Allocate 64 MB | 196.8 | 352.3 | Served from pre-reserved pool |
| `del` array | 132.8 | 352.3 | Returned to pool, not to OS |
| `free_all_blocks()` | 132.8 | 132.9 | Unused pool memory released to OS |

The pool pattern exists because `cudaMalloc` is slow. In a real workload where you allocate and free frequently (e.g. inside a training loop), the pool means only the first allocation pays the `cudaMalloc` cost — everything after that is just pointer arithmetic.


### Checking memory before large allocations

In [ ]:
device = cp.cuda.Device(0)
free_mem, total_mem = device.mem_info
print(f"Free:  {free_mem / 1024**3:.2f} GB")
print(f"Total: {total_mem / 1024**3:.2f} GB")
print(f"Used:  {(total_mem - free_mem) / 1024**3:.2f} GB")

Free:  14.32 GB
Total: 14.56 GB
Used:  0.24 GB


### Out-of-memory errors

In [ ]:
try:
    x = cp.zeros((100_000, 100_000), dtype=cp.float32)  # 40 GB — will fail
except cp.cuda.memory.OutOfMemoryError as e:
    print(f"OOM: {e}")
    pool.free_all_blocks()   # clean up and try smaller

OOM: Out of memory allocating 40,000,000,000 bytes (allocated so far: 139,308,032 bytes).


### Pinned memory for fast transfers

In [ ]:
# Regular transfer
a_pageable = np.zeros(1 << 25, dtype=np.float32)
t0 = time.perf_counter()
d = cp.asarray(a_pageable)
cp.cuda.Stream.null.synchronize()
t_regular = (time.perf_counter() - t0) * 1e3

# Pinned memory transfer (faster H→D)
a_pinned = cp.cuda.alloc_pinned_memory(a_pageable.nbytes)
a_pinned_view = np.frombuffer(a_pinned, dtype=np.float32)
a_pinned_view[:] = a_pageable

t0 = time.perf_counter()
d2 = cp.asarray(a_pinned_view)
cp.cuda.Stream.null.synchronize()
t_pinned = (time.perf_counter() - t0) * 1e3

print(f"Regular: {t_regular:.2f} ms")
print(f"Pinned:  {t_pinned:.2f} ms  ({t_regular/t_pinned:.1f}x faster)")

Regular: 54.30 ms
Pinned:  11.36 ms  (4.8x faster)


---

## 6. Custom Kernels in CuPy

When built-in ops aren't enough, CuPy provides three ways to write custom GPU code without leaving Python.

### 6.1 ElementwiseKernel — fastest option for element-wise ops

In [ ]:
# Syntax: ElementwiseKernel(inputs, outputs, operation, name)
leaky_relu = cp.ElementwiseKernel(
    'float32 x, float32 alpha',   # inputs (comma-separated)
    'float32 y',                   # output
    'y = (x > 0) ? x : alpha * x',# CUDA C expression
    'leaky_relu'                   # kernel name (for caching)
)

x = cp.random.randn(1_000_000).astype(cp.float32)
result = leaky_relu(x, 0.01)

# Verify
expected = cp.where(x > 0, x, 0.01 * x)
print("Max error:", float(cp.max(cp.abs(result - expected))))

Max error: 0.0


More complex example — fused multiply-add with clipping:

In [ ]:
fma_clip = cp.ElementwiseKernel(
    'float32 x, float32 y, float32 bias, float32 lo, float32 hi',
    'float32 out',
    '''
    float val = x * y + bias;
    out = fminf(fmaxf(val, lo), hi);
    ''',
    'fma_clip'
)

x = cp.random.rand(1000).astype(cp.float32)
y = cp.random.rand(1000).astype(cp.float32)
result = fma_clip(x, y, 0.1, 0.0, 1.0)
result[0:10]

array([0.31106287, 0.53462315, 0.31443137, 0.7443289 , 0.10599189,
       0.44837773, 0.65539306, 0.6615022 , 0.1300461 , 0.15509176],
      dtype=float32)

### 6.2 ReductionKernel — custom reductions

In [ ]:
# Custom reduction: sum of squares
sum_of_squares = cp.ReductionKernel(
    'float32 x',           # input
    'float32 out',         # output
    'x * x',               # map: transform each element
    'a + b',               # reduce: combine two elements
    'out = a',             # post-reduce: store result
    '0',                   # identity element
    'sum_of_squares'       # name
)

x = cp.array([1., 2., 3., 4.], dtype=cp.float32)
print(sum_of_squares(x))   # 1+4+9+16 = 30

30.0


### 6.3 RawKernel — full CUDA C++ kernel as a string

When you need shared memory, synchronization, or any pattern that ElementwiseKernel can't express:

In [ ]:
# This is a complete CUDA C kernel written as a Python string
reduction_code = r'''
extern "C" __global__
void block_reduce(const float* input, float* output, int n) {
    __shared__ float sdata[256];

    int tid = threadIdx.x;
    int i   = blockIdx.x * blockDim.x + tid;

    sdata[tid] = (i < n) ? input[i] : 0.0f;
    __syncthreads();

    for (int stride = blockDim.x / 2; stride > 0; stride >>= 1) {
        if (tid < stride)
            sdata[tid] += sdata[tid + stride];
        __syncthreads();
    }

    if (tid == 0)
        atomicAdd(output, sdata[0]);
}
'''

# Compile the kernel
reduce_kernel = cp.RawKernel(reduction_code, 'block_reduce')

N = 1 << 22
x = cp.ones(N, dtype=cp.float32)
out = cp.zeros(1, dtype=cp.float32)

block = 256
grid = (N + block - 1) // block
reduce_kernel((grid,), (block,), (x, out, N))
cp.cuda.Stream.null.synchronize()

print(f"Sum: {float(out[0]):.0f}  Expected: {N}")

Sum: 4194304  Expected: 4194304


> **When to use each:**
> - `ElementwiseKernel` — custom element-wise formula, simple and fast
> - `ReductionKernel` — custom reduction (e.g., sum of abs, weighted sum)
> - `RawKernel` — anything needing shared memory, full control

---

## 7. SciPy on GPU — cupyx.scipy

`cupyx.scipy` mirrors `scipy` on the GPU. Most of scipy's ndimage, signal, linalg, and sparse APIs are available.

In [ ]:
from cupyx.scipy import ndimage as cpnd
from cupyx.scipy import signal as cpsig
from cupyx.scipy import linalg as cpla
import scipy.ndimage as spnd

# --- Image filtering ---
img = cp.random.rand(2048, 2048).astype(cp.float32)

# Gaussian blur
blurred = cpnd.gaussian_filter(img, sigma=2.0)

# Sobel edge detection
gx = cpnd.sobel(img, axis=1)
gy = cpnd.sobel(img, axis=0)
edges = cp.sqrt(gx**2 + gy**2)

# --- Signal processing ---
signal = cp.random.rand(100_000).astype(cp.float32)
# FIR filter with scipy-designed coefficients
import scipy.signal
b = scipy.signal.firwin(101, 0.2)   # design on CPU
b_gpu = cp.asarray(b.astype(np.float32))
filtered = cpsig.fftconvolve(signal, b_gpu)

# --- Linear algebra ---
A = cp.random.rand(1000, 1000).astype(cp.float32)
b = cp.random.rand(1000).astype(cp.float32)

x = cp.linalg.solve(A, b)

# Verify: A @ x should equal b
residual = cp.linalg.norm(A @ x - b)
print("Residual:", residual)  # should be near 0

Residual: 0.0027084695


---

## 8. FFT and Linear Algebra

These are the two areas where CuPy provides the most dramatic speedups, because it calls cuFFT and cuBLAS directly.

### FFT

In [ ]:
# 1D FFT
N = 1 << 20
x = cp.random.rand(N).astype(cp.float32)

X = cp.fft.fft(x)           # complex FFT
X_r = cp.fft.rfft(x)        # real FFT (N/2+1 outputs)
freqs = cp.fft.fftfreq(N)   # frequency bins

# 2D FFT (e.g., image processing)
img = cp.random.rand(1024, 1024).astype(cp.float32)
IMG = cp.fft.fft2(img)
img_back = cp.fft.ifft2(IMG).real

# Benchmark: NumPy vs CuPy FFT
import numpy as np
x_np = x.get()
t_np = bench_cpu(np.fft.fft, x_np)
t_cp = bench_gpu_cp(cp.fft.fft, x)
print(f"1D FFT ({N:,}): NumPy={t_np:.1f}ms  CuPy={t_cp:.1f}ms  ({t_np/t_cp:.1f}x)")

1D FFT (1,048,576): NumPy=58.6ms  CuPy=0.4ms  (147.8x)


### Linear Algebra

In [ ]:
# Matrix multiply — calls cuBLAS sgemm
A = cp.random.rand(4096, 4096).astype(cp.float32)
B = cp.random.rand(4096, 4096).astype(cp.float32)
C = cp.dot(A, B)           # or A @ B

# Singular Value Decomposition
A_small = cp.random.rand(1000, 500).astype(cp.float32)
U, s, Vt = cp.linalg.svd(A_small, full_matrices=False)
print(f"U: {U.shape}, s: {s.shape}, Vt: {Vt.shape}")

# Eigendecomposition
A_sym = A_small @ A_small.T
eigenvalues, eigenvectors = cp.linalg.eigh(A_sym)

# Least squares
b = cp.random.rand(1000).astype(cp.float32)
x, residuals, rank, sv = cp.linalg.lstsq(A_small, b, rcond=None)

U: (1000, 500), s: (500,), Vt: (500, 500)


---

## 9. Interoperability

### With NumPy

In [ ]:
a_np = np.array([1., 2., 3.])
a_cp = cp.asarray(a_np)          # np → cp

b_cp = cp.array([4., 5., 6.])
b_np = b_cp.get()                # cp → np

# NEVER mix in one operation
try:
    _ = a_np + b_cp
except TypeError as e:
    print(f"Error: {e}")

Error: Unsupported type <class 'numpy.ndarray'>


### With Numba

CuPy and Numba share GPU memory through the `__cuda_array_interface__` protocol. No copying:

In [ ]:
from numba import cuda
from numba import float32

@cuda.jit
def scale_inplace(arr, factor):
    i = cuda.grid(1)
    if i < arr.shape[0]:
        arr[i] *= factor

x = cp.ones(1024, dtype=cp.float32) * 3.0

# ✅ Cast factor to float32 explicitly — matches the CuPy array dtype
scale_inplace[4, 256](x, float32(2.0))
cuda.synchronize()
print(x[:5])  # [6., 6., 6., 6., 6.]

[6. 6. 6. 6. 6.]


### With PyTorch

In [ ]:
import torch

# CuPy → PyTorch (zero-copy via DLPack)
x_cp = cp.random.rand(10).astype(cp.float32)
x_torch = torch.as_tensor(x_cp, device='cuda')   # zero-copy!

# PyTorch → CuPy (zero-copy)
t = torch.randn(10, device='cuda')
x_cp2 = cp.asarray(t)    # zero-copy

print("Same memory:", x_cp2.data.ptr == t.data_ptr())

---

## 10. Common Pitfalls

### Pitfall 1: Forgetting to synchronize when timing

In [ ]:
# WRONG — measures Python overhead, not GPU time
t0 = time.perf_counter()
result = cp.sum(x)
print(time.perf_counter() - t0)   # meaninglessly small

# CORRECT
cp.cuda.Stream.null.synchronize()
t0 = time.perf_counter()
result = cp.sum(x)
cp.cuda.Stream.null.synchronize()
print(time.perf_counter() - t0)

0.00044502399941848125
0.0003544709998095641


### Pitfall 2: Scalar extraction in a loop

In [ ]:
# 2D array: 1000 rows, each row is what we want to sum
data = cp.random.rand(1000, 10000).astype(cp.float32)

# ── SLOW: each .item() forces a device→host sync ──
t0 = time.perf_counter()
results_slow = []
for i in range(1000):
    val = cp.sum(data[i]).item()   # 1000 synchronizations!
    results_slow.append(val)
t_slow = (time.perf_counter() - t0) * 1e3

# ── FAST: batch the computation, transfer once ──
cp.cuda.Stream.null.synchronize()
t0 = time.perf_counter()
results_fast = cp.sum(data, axis=1).get()   # one kernel + one transfer
cp.cuda.Stream.null.synchronize()
t_fast = (time.perf_counter() - t0) * 1e3

print(f"Slow (1000 syncs):  {t_slow:.1f} ms")
print(f"Fast (1 kernel):    {t_fast:.1f} ms  ({t_slow/t_fast:.1f}x speedup)")

# Verify same results
print("Results match:", np.allclose(results_slow, results_fast, rtol=1e-4))

Slow (1000 syncs):  54.3 ms
Fast (1 kernel):    0.4 ms  (145.3x speedup)
Results match: True


### Pitfall 3: Mixing dtypes silently

In [ ]:
a = cp.ones(10, dtype=cp.float32)
b = cp.ones(10, dtype=cp.float64)
c = a + b   # promotes to float64 — might not be what you want
print(c.dtype)   # float64 — and float64 is 2x slower on most GPUs

float64


### Pitfall 4: Unnecessary transfers in a loop

In [ ]:
x_gpu = cp.ones(1 << 22, dtype=cp.float32)  # 4M elements

# ── SLOW: moves data back and forth every iteration ──
t0 = time.perf_counter()
for _ in range(100):
    x_gpu = cp.asarray(x_gpu.get())    # H→D
    x_gpu = x_gpu * 2                  # GPU compute
    x_cpu = x_gpu.get()                # D→H
t_slow = (time.perf_counter() - t0) * 1e3

# Reset
x_gpu = cp.ones(1 << 22, dtype=cp.float32)

# ── FAST: keep data on GPU ──
cp.cuda.Stream.null.synchronize()
t0 = time.perf_counter()
for _ in range(100):
    x_gpu = x_gpu * 2            # stays on GPU
cp.cuda.Stream.null.synchronize()
t_fast = (time.perf_counter() - t0) * 1e3

print(f"Slow (100 roundtrips): {t_slow:.1f} ms")
print(f"Fast (GPU only):       {t_fast:.1f} ms  ({t_slow/t_fast:.1f}x speedup)")

Slow (100 roundtrips): 1467.5 ms
Fast (GPU only):       15.3 ms  (95.9x speedup)


---

## 11. Exercises

### Exercise 1 — Portfolio Analytics (Medium)

Given a matrix of daily returns `R[days × assets]` (float32, 1000 days, 5000 assets):

1. Compute the mean return for each asset (shape: `(5000,)`)
2. Compute the covariance matrix (shape: `(5000, 5000)`)
3. Find the 10 assets with highest Sharpe ratio: `mean / std`
4. Benchmark all steps against NumPy

In [ ]:
# Starter code
import cupy as cp
import numpy as np

np.random.seed(42)
returns_np = np.random.randn(1000, 5000).astype(np.float32) * 0.01

# Your code here
returns_cp = cp.asarray(returns_np)

### Exercise 2 — Image Batch Processing (Medium)

You have a batch of 100 grayscale images, each 512×512.

1. Apply Gaussian blur to all 100 images simultaneously (hint: use 3D array `[100, 512, 512]`)
2. Normalize each image to zero mean and unit variance
3. Compute a histogram of all pixel values across the batch (256 bins)
4. Find the top-10 brightest images (highest mean value)

### Exercise 3 — Custom Kernel (Hard)

Implement a **softmax** function using `ReductionKernel` and `ElementwiseKernel`:

$$\text{softmax}(x_i) = \frac{e^{x_i - \max(x)}}{\sum_j e^{x_j - \max(x)}}$$

Steps:
1. Use `ReductionKernel` to find `max(x)`
2. Use `ElementwiseKernel` to compute `exp(x - max)`
3. Use `ReductionKernel` to compute the sum
4. Use `ElementwiseKernel` to divide

Verify against `cp.exp(x) / cp.sum(cp.exp(x))` and benchmark both.

## CuPy — Official Documentation & Reference Resources

### Official Documentation (start here)

| Resource | URL | What it covers |
|----------|-----|----------------|
| **CuPy Docs — User Guide** | https://docs.cupy.dev/en/stable/user_guide/index.html | Concepts, memory model, custom kernels, interop |
| **CuPy Docs — API Reference** | https://docs.cupy.dev/en/stable/reference/index.html | Every function, class, and parameter |
| **CuPy Docs — Overview** | https://docs.cupy.dev/en/stable/overview.html | High-level introduction |

The current stable version is **14.1.0** (released May 23, 2026). Make sure you're reading the docs that match your installed version. You can check with:

```python
import cupy as cp
print(cp.__version__)
```

---

### Source and Examples

| Resource | URL | What it covers |
|----------|-----|----------------|
| **GitHub repo** | https://github.com/cupy/cupy | Source code, examples folder, issue tracker |
| **NumPy docs** | https://numpy.org/doc/stable/ | Since CuPy mirrors NumPy — if a function exists in NumPy, it almost certainly works the same in CuPy |

CuPy's GitHub also links directly to tutorials and examples, which are useful for seeing real usage patterns.

---

### How to Look Things Up Efficiently

**"Does CuPy have X?"** — search the API reference, or just try:

```python
import cupy as cp
help(cp.your_function)     # inline docs
dir(cp.linalg)             # list everything in a submodule
```

**"Does CuPy support this NumPy function?"** — check the compatibility table:
https://docs.cupy.dev/en/stable/reference/comparison.html

This page lists every NumPy/SciPy function and whether CuPy supports it — very useful when porting existing code.